<div style="
    padding: 15px 20px;
    margin: 10px 0;
    border-left: 8px solid #4F7942;
    background-color: rgba(255, 191, 0, 0.05);
    border-radius: 4px;
">

## Table of Contents:

- [Imports](#imports)
- [Load Sample Signal](#load-sample-signal)
- [Extract IQ Stats And Metrics](#extract-iq-stats-and-metrics)
- [Extract Magnitude Stats And Metrics](#extract-magnitude-stats-and-metrics)
- [Extract Power Stats And Metrics](#extract-power-stats-and-metrics)
- [Extract Phase Stats And Metrics](#extract-phase-stats-and-metrics)
- [Extract Frequency Stats And Metrics](#extract-frequency-stats-and-metrics)
- [Extract FFT Stats And Metrics](#extract-fft-stats-and-metrics)
- [Extract Spectral Stats And Metrics](#extract-spectral-stats-and-metrics)
- [Extract Constellation Stats And Metrics](#extract-constellation-stats-and-metrics)
- [Feature Engineered Dataset Creation](#feature-engineered-dataset-creation)
- [Feature Engineered Dataset Saving](#feature-engineered-dataset-saving)
- [Feature Engineered Dataset Shape](#feature-engineered-dataset-shape)

</div>

##### **IMPORTANT**:
- YOU MUST HAVE THE REDUCED DATASET `../Datasets/highest_snr_reduced_df.hdf5` before running this notebook.
- YOU MUST HAVE the helper module `highest_snr_feature_engineering.py` available in the same project environment.
- The feature-engineered output is tabular metadata/features, not the raw `(1024, 2)` I/Q signal tensor.
- If the full run is slow, set `MAX_SIGNALS = 100` in the sample-loading section for a quick dry run, then change it back to `None` for the full dataset.

---

##### **ORIENTATION**:
- The baseline notebook used flattened raw I/Q data directly with a Random Forest Classifier. This notebook creates a more interpretable tabular dataset from the same reduced highest-SNR data.
- Each signal becomes one row. Each column becomes a signal-derived feature such as I/Q statistics, magnitude behavior, power behavior, phase behavior, frequency behavior, FFT/spectral behavior, and constellation geometry.
- The saved feature-engineered dataset can then be used for traditional ML experiments and compared against the flattened raw I/Q baseline.

---

##### **SUMMARY**:
- Load Sample Signal
    - Pulls one signal frame from the reduced highest-SNR dataset so each feature group can be inspected before processing the full dataset
- Extract IQ Stats And Metrics
    - Extracts direct statistics from the I and Q channels
- Extract Magnitude Stats And Metrics
    - Extracts amplitude/radius behavior from complex I/Q
- Extract Power Stats And Metrics
    - Extracts power, energy, and peak-to-average behavior
- Extract Phase Stats And Metrics
    - Extracts wrapped and unwrapped phase behavior
- Extract Frequency Stats And Metrics
    - Extracts instantaneous-frequency-style behavior from phase differences
- Extract FFT Stats And Metrics
    - Extracts frequency-domain FFT magnitude and power summaries
- Extract Spectral Stats And Metrics
    - Extracts spectral centroid, spread, entropy, flatness, rolloff, and bandwidth-style features
- Extract Constellation Stats And Metrics
    - Extracts I/Q scatter-plot geometry features
- Feature Engineered Dataset Creation
    - Runs all feature engineering functions over the reduced dataset
- Feature Engineered Dataset Saving
    - Saves the feature-engineered dataset as parquet and HDF5
- Feature Engineered Dataset Shape
    - Verifies dataset shape, class distribution, SNR distribution, and ML-ready feature/target splits

<div style="
    padding: 15px 20px;
    margin: 10px 0;
    border-left: 8px solid #4F7942;
    background-color: rgba(255, 191, 0, 0.05);
    border-radius: 4px;
">

## Imports

- [Back to Table of Contents](#table-of-contents)

</div>

In [ ]:
# Helper functions to help take function output and show what is happening
import numpy as np
import pandas as pd
import h5py

# Primary functions for this file
import highest_snr_feature_engineering as fe

# Pandas configurations
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

<div style="
    padding: 15px 20px;
    margin: 10px 0;
    border-left: 8px solid #4F7942;
    background-color: rgba(255, 191, 0, 0.05);
    border-radius: 4px;
">

## Load Sample Signal

- [Back to Table of Contents](#table-of-contents)

</div>

##### **SUMMARY**:
- Loads one signal from the reduced highest-SNR dataset
- This sample is used to validate each feature-engineering function before processing the full dataset
- `MAX_SIGNALS = None` processes the full dataset later; set it to a small integer for quick testing

In [ ]:
# Simply just loads a signal and prints the details
SAMPLE_INDEX = 0
MAX_SIGNALS = None

with h5py.File(fe.HIGHEST_SNR_REDUCED_DF, "r") as f:
    sample_signal = f["X"][SAMPLE_INDEX]
    sample_label = f["Y"][SAMPLE_INDEX]
    sample_snr = f["Z"][SAMPLE_INDEX][0]

sample_mod_id = int(np.argmax(sample_label))
sample_mod_type = fe.MOD_TYPE_MAPPING[sample_mod_id]

print("Sample index:", SAMPLE_INDEX)
print("Sample signal shape:", sample_signal.shape)
print("Sample modulation ID:", sample_mod_id)
print("Sample modulation type:", sample_mod_type)
print("Sample SNR:", sample_snr)

Sample index: 0
Sample signal shape: (1024, 2)
Sample modulation ID: 0
Sample modulation type: OOK
Sample SNR: 30


<div style="
    padding: 15px 20px;
    margin: 10px 0;
    border-left: 8px solid #4F7942;
    background-color: rgba(255, 191, 0, 0.05);
    border-radius: 4px;
">

## Extract IQ Stats And Metrics

- [Back to Table of Contents](#table-of-contents)

</div>

##### **SUMMARY**:
- Extracts direct statistics from the I and Q channels
- These features summarize the raw in-phase and quadrature distributions
- Useful for simple baseline models because they capture central tendency, spread, range, skew, kurtosis, and I/Q correlation

In [ ]:
# Extract I/Q features
iq_features = fe.extract_basic_iq_features(sample_signal)

df_iq_features = pd.DataFrame(
    iq_features.items(),
    columns=["feature", "value"]
)

display(df_iq_features)

,feature,value
0,mean_I,0.739839
1,std_I,0.781117
2,min_I,-0.500421
3,max_I,2.351720
4,median_I,0.559564
5,range_I,2.852141
6,q25_I,0.016489
7,q75_I,1.457256
8,iqr_I,1.440768
9,skew_I,0.323544


<div style="
    padding: 15px 20px;
    margin: 10px 0;
    border-left: 8px solid #4F7942;
    background-color: rgba(255, 191, 0, 0.05);
    border-radius: 4px;
">

## Extract Magnitude Stats And Metrics

- [Back to Table of Contents](#table-of-contents)

</div>

##### **SUMMARY**:
- Extracts magnitude/amplitude features from complex I/Q
- Magnitude is calculated as `sqrt(I^2 + Q^2)`
- Helpful because ASK, QAM, OOK, and other amplitude-sensitive modulations can differ in radius behavior

In [ ]:
# Extract magnitude features
magnitude_power_features = fe.extract_magnitude_power_features(sample_signal)

magnitude_features = {
    feature_name: feature_value
    for feature_name, feature_value in magnitude_power_features.items()
    if "magnitude" in feature_name
}

df_magnitude_features = pd.DataFrame(
    magnitude_features.items(),
    columns=["feature", "value"]
)

display(df_magnitude_features)

,feature,value
0,mean_magnitude,0.990590
1,std_magnitude,0.845569
2,min_magnitude,0.000709
3,max_magnitude,2.776072
4,median_magnitude,0.679575
5,range_magnitude,2.775363
6,q25_magnitude,0.225010
7,q75_magnitude,1.780160
8,iqr_magnitude,1.555150
9,skew_magnitude,0.475403


<div style="
    padding: 15px 20px;
    margin: 10px 0;
    border-left: 8px solid #4F7942;
    background-color: rgba(255, 191, 0, 0.05);
    border-radius: 4px;
">

## Extract Power Stats And Metrics

- [Back to Table of Contents](#table-of-contents)

</div>

##### **SUMMARY**:
- Extracts power and energy features from complex I/Q
- Power is calculated as `magnitude^2`
- Helpful for identifying modulations that differ in energy concentration, peak-to-average behavior, or amplitude stability

In [ ]:
# Extract magnitude power features
magnitude_power_features = fe.extract_magnitude_power_features(sample_signal)

power_features = {
    feature_name: feature_value
    for feature_name, feature_value in magnitude_power_features.items()
    if (
        "power" in feature_name
        or "energy" in feature_name
        or "rms" in feature_name
        or "peak_to_average" in feature_name
    )
}

df_power_features = pd.DataFrame(
    power_features.items(),
    columns=["feature", "value"]
)

display(df_power_features)

,feature,value
0,mean_power,1.696255e+00
1,std_power,2.084705e+00
2,min_power,5.021242e-07
3,max_power,7.706573e+00
4,median_power,4.618230e-01
5,total_energy,1.736965e+03
6,rms_power,1.302404e+00
7,peak_to_average_power_ratio,4.543287e+00
8,power_cv,1.229004e+00


<div style="
    padding: 15px 20px;
    margin: 10px 0;
    border-left: 8px solid #4F7942;
    background-color: rgba(255, 191, 0, 0.05);
    border-radius: 4px;
">

## Extract Phase Stats And Metrics

- [Back to Table of Contents](#table-of-contents)

</div>

##### **SUMMARY**:
- Extracts wrapped phase, unwrapped phase, and circular phase summaries
- Phase is calculated from complex I/Q using `angle(I + jQ)`
- Helpful because PSK and FM-style modulations often differ strongly in phase behavior

In [ ]:
# Extract phase frequency features
phase_frequency_features = fe.extract_phase_frequency_features(sample_signal)

phase_features = {
    feature_name: feature_value
    for feature_name, feature_value in phase_frequency_features.items()
    if "phase" in feature_name or "circular" in feature_name
}

df_phase_features = pd.DataFrame(
    phase_features.items(),
    columns=["feature", "value"]
)

display(df_phase_features)

,feature,value
0,mean_phase,-0.109794
1,std_phase,1.330908
2,min_phase,-2.937152
3,max_phase,3.103870
4,range_phase,6.041022
5,circular_mean_phase,0.607782
6,circular_resultant_length,0.537871
7,mean_unwrapped_phase,18.856350
8,std_unwrapped_phase,7.840054
9,unwrapped_phase_range,37.833401


<div style="
    padding: 15px 20px;
    margin: 10px 0;
    border-left: 8px solid #4F7942;
    background-color: rgba(255, 191, 0, 0.05);
    border-radius: 4px;
">

## Extract Frequency Stats And Metrics

- [Back to Table of Contents](#table-of-contents)

</div>

##### **SUMMARY**:
- Extracts instantaneous-frequency-style features from phase differences
- This approximates local frequency changes using differences in unwrapped phase
- Helpful for modulations where frequency or phase transition behavior is distinctive

In [ ]:
# Extract frequency features
phase_frequency_features = fe.extract_phase_frequency_features(sample_signal)

frequency_features = {
    feature_name: feature_value
    for feature_name, feature_value in phase_frequency_features.items()
    if "freq" in feature_name
}

df_frequency_features = pd.DataFrame(
    frequency_features.items(),
    columns=["feature", "value"]
)

display(df_frequency_features)

,feature,value
0,mean_inst_freq,0.016814
1,std_inst_freq,0.704658
2,min_inst_freq,-3.140634
3,max_inst_freq,3.140934
4,median_inst_freq,0.000107
5,range_inst_freq,6.281568
6,q25_inst_freq,-0.000887
7,q75_inst_freq,0.001273
8,iqr_inst_freq,0.002161
9,skew_inst_freq,0.423945


<div style="
    padding: 15px 20px;
    margin: 10px 0;
    border-left: 8px solid #4F7942;
    background-color: rgba(255, 191, 0, 0.05);
    border-radius: 4px;
">

## Extract FFT Stats And Metrics

- [Back to Table of Contents](#table-of-contents)

</div>

##### **SUMMARY**:
- Extracts FFT magnitude and FFT power summary features
- FFT converts each signal from time-domain I/Q into frequency-domain information
- Helpful for modulations with distinctive frequency-domain structure

In [ ]:
# Extract FFT features
fft_spectral_features = fe.extract_fft_spectral_features(sample_signal)

fft_features = {
    feature_name: feature_value
    for feature_name, feature_value in fft_spectral_features.items()
    if "fft" in feature_name
}

df_fft_features = pd.DataFrame(
    fft_features.items(),
    columns=["feature", "value"]
)

display(df_fft_features)

,feature,value
0,mean_fft_magnitude,1.112008e+01
1,std_fft_magnitude,4.016602e+01
2,min_fft_magnitude,1.567703e-02
3,max_fft_magnitude,9.183599e+02
4,median_fft_magnitude,3.852096e-01
5,skew_fft_magnitude,1.239265e+01
6,kurtosis_fft_magnitude,2.546628e+02
7,mean_fft_power,1.736965e+03
8,std_fft_power,2.654338e+04
9,max_fft_power,8.433848e+05


<div style="
    padding: 15px 20px;
    margin: 10px 0;
    border-left: 8px solid #4F7942;
    background-color: rgba(255, 191, 0, 0.05);
    border-radius: 4px;
">

## Extract Spectral Stats And Metrics

- [Back to Table of Contents](#table-of-contents)

</div>

##### **SUMMARY**:
- Extracts spectral centroid, spread, entropy, flatness, rolloff, dominant frequency, and occupied-bandwidth-style features
- These summarize where energy is located across normalized frequency
- Helpful for OFDM, FM, and other spectrally distinctive modulation families

In [ ]:
# Extract Spectral features
fft_spectral_features = fe.extract_fft_spectral_features(sample_signal)

spectral_features = {
    feature_name: feature_value
    for feature_name, feature_value in fft_spectral_features.items()
    if (
        "spectral" in feature_name
        or "bandwidth" in feature_name
        or "dominant_freq" in feature_name
        or "rolloff" in feature_name
    )
}

df_spectral_features = pd.DataFrame(
    spectral_features.items(),
    columns=["feature", "value"]
)

display(df_spectral_features)

,feature,value
0,spectral_centroid,0.000024
1,spectral_spread,0.024246
2,spectral_entropy,4.397296
3,spectral_flatness,0.000278
4,dominant_freq,0.000000
5,spectral_rolloff_85_freq,0.023438
6,spectral_rolloff_95_freq,0.044922
7,occupied_bandwidth_90,0.089844


<div style="
    padding: 15px 20px;
    margin: 10px 0;
    border-left: 8px solid #4F7942;
    background-color: rgba(255, 191, 0, 0.05);
    border-radius: 4px;
">

## Extract Constellation Stats And Metrics

- [Back to Table of Contents](#table-of-contents)

</div>

##### **SUMMARY**:
- Extracts constellation-style geometry features from the I/Q scatter plot
- Includes covariance, quadrant distribution, radius behavior, and rounded radius/phase uniqueness
- Helpful for ASK, PSK, QAM, and other modulations where the constellation shape is meaningful

In [ ]:
# Extract constellation features
constellation_features = fe.extract_constellation_features(sample_signal)

df_constellation_features = pd.DataFrame(
    constellation_features.items(),
    columns=["feature", "value"]
)

display(df_constellation_features)

,feature,value
0,cov_I_I,0.610740
1,cov_I_Q,0.411441
2,cov_Q_I,0.411441
3,cov_Q_Q,0.282073
4,constellation_area_proxy,0.414653
5,quadrant_1_ratio,0.768555
6,quadrant_2_ratio,0.002930
7,quadrant_3_ratio,0.228516
8,quadrant_4_ratio,0.000000
9,quadrant_balance_std,0.313410


<div style="
    padding: 15px 20px;
    margin: 10px 0;
    border-left: 8px solid #4F7942;
    background-color: rgba(255, 191, 0, 0.05);
    border-radius: 4px;
">

## Feature Engineered Dataset Creation

- [Back to Table of Contents](#table-of-contents)

</div>

##### **SUMMARY**:
- Runs all feature-engineering functions over the reduced highest-SNR dataset
- Each signal becomes one row in a pandas DataFrame
- Metadata columns are kept at the front: `signal_index`, `modulation_id`, `modulation_type`, and `snr`

In [ ]:
# Create the full feature engineered dataset
df_features = fe.create_feature_engineered_dataset(
    hdf5_data_filepath=fe.HIGHEST_SNR_REDUCED_DF,
    mod_type_mapping=fe.MOD_TYPE_MAPPING,
    max_signals=MAX_SIGNALS,
    print_progress=True
)

display(df_features.head())

FEATURE ENGINEERING STARTED
Dataset filepath: ../Datasets/highest_snr_reduced_df.hdf5
Signals to process: 24000
X shape: (24000, 1024, 2)
Y shape: (24000, 24)
Z shape: (24000, 1)

Processed 1000 / 24000 signals...
Processed 2000 / 24000 signals...
Processed 3000 / 24000 signals...
Processed 4000 / 24000 signals...
Processed 5000 / 24000 signals...
Processed 6000 / 24000 signals...
Processed 7000 / 24000 signals...
Processed 8000 / 24000 signals...
Processed 9000 / 24000 signals...
Processed 10000 / 24000 signals...
Processed 11000 / 24000 signals...
Processed 12000 / 24000 signals...
Processed 13000 / 24000 signals...
Processed 14000 / 24000 signals...
Processed 15000 / 24000 signals...
Processed 16000 / 24000 signals...
Processed 17000 / 24000 signals...
Processed 18000 / 24000 signals...
Processed 19000 / 24000 signals...
Processed 20000 / 24000 signals...
Processed 21000 / 24000 signals...
Processed 22000 / 24000 signals...
Processed 23000 / 24000 signals...

FEATURE ENGINEERING COM

,signal_index,modulation_id,modulation_type,snr,mean_I,std_I,min_I,max_I,median_I,range_I,q25_I,q75_I,iqr_I,skew_I,kurtosis_I,mean_Q,std_Q,min_Q,max_Q,median_Q,range_Q,q25_Q,q75_Q,iqr_Q,skew_Q,kurtosis_Q,mean_abs_I,mean_abs_Q,std_ratio_I_Q,mean_abs_ratio_I_Q,corr_I_Q,mean_magnitude,std_magnitude,min_magnitude,max_magnitude,median_magnitude,range_magnitude,q25_magnitude,q75_magnitude,iqr_magnitude,skew_magnitude,kurtosis_magnitude,mean_power,std_power,min_power,max_power,median_power,total_energy,rms_power,peak_to_average_power_ratio,magnitude_cv,power_cv,mean_phase,std_phase,min_phase,max_phase,range_phase,circular_mean_phase,circular_resultant_length,mean_unwrapped_phase,std_unwrapped_phase,unwrapped_phase_range,mean_inst_freq,std_inst_freq,min_inst_freq,max_inst_freq,median_inst_freq,range_inst_freq,q25_inst_freq,q75_inst_freq,iqr_inst_freq,skew_inst_freq,kurtosis_inst_freq,mean_fft_magnitude,std_fft_magnitude,min_fft_magnitude,max_fft_magnitude,median_fft_magnitude,skew_fft_magnitude,kurtosis_fft_magnitude,mean_fft_power,std_fft_power,max_fft_power,total_fft_power,spectral_centroid,spectral_spread,spectral_entropy,spectral_flatness,dominant_freq,peak_fft_power_ratio,spectral_rolloff_85_freq,spectral_rolloff_95_freq,occupied_bandwidth_90,cov_I_I,cov_I_Q,cov_Q_I,cov_Q_Q,constellation_area_proxy,quadrant_1_ratio,quadrant_2_ratio,quadrant_3_ratio,quadrant_4_ratio,quadrant_balance_std,near_origin_ratio,far_origin_ratio,radius_mean,radius_std,radius_unique_rounded_count,phase_unique_rounded_count
0,0,0,OOK,30,0.739839,0.781117,-0.500421,2.351720,0.559564,2.852141,0.016489,1.457256,1.440768,0.323544,-1.242235,0.506905,0.530846,-0.393557,1.572671,0.393726,1.966227,0.010876,1.010403,0.999527,0.285936,-1.299135,0.816420,0.559229,1.471457,1.459902,0.991287,0.990590,0.845569,0.000709,2.776072,0.679575,2.775363,0.225010,1.780160,1.555150,0.475403,-1.216012,1.696255,2.084705,5.021242e-07,7.706573,0.461823,1736.965210,1.302404,4.543287,0.853601,1.229004,-0.109794,1.330908,-2.937152,3.103870,6.041022,0.607782,0.537871,18.856350,7.840054,37.833401,0.016814,0.704658,-3.140634,3.140934,0.000107,6.281568,-0.000887,0.001273,0.002161,0.423945,15.776016,11.120085,40.166016,0.015677,918.359863,0.385210,12.392648,254.662811,1736.965332,26543.375000,8.433848e+05,1778652.50,0.000024,0.024246,4.397296,0.000278,0.0,0.474171,0.023438,0.044922,0.089844,0.610740,0.411441,0.411441,0.282073,0.414653,0.768555,0.002930,0.228516,0.000000,0.313410,0.567383,0.230469,0.990590,0.845569,241,54
1,1,0,OOK,30,1.011886,1.002867,-0.734198,2.919468,0.918309,3.653667,0.078252,1.958826,1.880574,0.141994,-1.378221,-0.019308,0.033715,-0.137604,0.024156,-0.005056,0.161760,-0.031092,0.001897,0.032990,-1.501081,1.616800,1.109965,0.023951,29.745008,46.342579,-0.575323,1.110384,0.893427,0.001663,2.919661,0.919164,2.917998,0.239035,1.959959,1.720923,0.317480,-1.382836,2.031164,2.309860,2.766188e-06,8.524419,0.844862,2079.912109,1.425189,4.196815,0.804610,1.137210,0.451268,1.348481,-3.141350,3.141257,6.282607,-0.018225,0.582017,-4.837897,6.640921,25.169464,0.003007,0.652751,-3.140761,3.141482,-0.000053,6.282243,-0.000911,0.000736,0.001648,0.139158,19.739037,12.434528,43.878178,0.784873,1036.359619,1.538881,13.592319,290.584595,2079.912109,33805.968750,1.074041e+06,2129830.00,-0.000013,0.025126,4.101415,0.003429,0.0,0.504285,0.024414,0.043945,0.087891,1.006725,-0.019472,-0.019472,0.001138,0.033812,0.150391,0.179688,0.029297,0.640625,0.232467,0.535156,0.234375,1.110384,0.893427,242,28
2,2,0,OOK,30,0.811902,0.713866,-0.462071,2.154327,0.836580,2.616398,0.149772,1.465518,1.315745,-0.023391,-1.360287,0.798254,0.702855,-0.420834,2.130226,0.823946,2.551059,0.144123,1.462950,1.318827,-0.028206,-1.361135,0.874570,0.861073,1.015666,1.015675,0.971862,1.231032,0.885757,0.003267,2.771781,1.209571,2.768514,0.347216,2.074491,1.727275,0.119565,-1.521644,2.300004,2.336544,1.067332e-05,7.682770,1.463062,2355.204590,1.516577,3.340328,0.719524,1.015887,0.216069,1.205052,-2.493611,0.923082,3.416693,0.7

<div style="
    padding: 15px 20px;
    margin: 10px 0;
    border-left: 8px solid #4F7942;
    background-color: rgba(255, 191, 0, 0.05);
    border-radius: 4px;
">

## Feature Engineered Dataset Saving

- [Back to Table of Contents](#table-of-contents)

</div>

##### **SUMMARY**:
- Saves the feature-engineered dataset for later model training/testing
- Parquet is generally best for pandas tabular data
- HDF5 is also saved to stay consistent with the rest of the project

In [ ]:
# Save off the full feature engineered dataset
fe.save_feature_engineered_dataset(
    df_features=df_features,
    output_filepath=fe.FEATURE_ENGINEERED_DATASET_PARQUET
)

FEATURE-ENGINEERED DATASET SAVED
Output filepath: ../Datasets/highest_snr_feature_engineered.parquet
Dataset shape: (24000, 109)



<div style="
    padding: 15px 20px;
    margin: 10px 0;
    border-left: 8px solid #4F7942;
    background-color: rgba(255, 191, 0, 0.05);
    border-radius: 4px;
">

## Feature Engineered Dataset Shape

- [Back to Table of Contents](#table-of-contents)

</div>

##### **SUMMARY**:
- Confirms the saved feature-engineered dataset shape
- Separates metadata columns from model feature columns
- Creates `X_features` and `y_labels` variables that can be reused for later ML training/testing

In [ ]:
# Show the outline of the feature engineered dataset to verify correctness
metadata_cols = [
    "signal_index",
    "modulation_id",
    "modulation_type",
    "snr"
]

feature_cols = [
    col for col in df_features.columns
    if col not in metadata_cols
]

X_features = df_features[feature_cols]
y_labels = df_features["modulation_id"]

print("Feature-engineered dataset shape:", df_features.shape)
print("X_features shape:", X_features.shape)
print("y_labels shape:", y_labels.shape)
print("Number of metadata columns:", len(metadata_cols))
print("Number of feature columns:", len(feature_cols))
print()

print("Modulation distribution:")
display(
    df_features["modulation_type"]
    .value_counts()
    .rename_axis("modulation_type")
    .reset_index(name="num_signals")
)

print("SNR distribution:")
display(
    df_features["snr"]
    .value_counts()
    .sort_index()
    .rename_axis("snr")
    .reset_index(name="num_signals")
)

print("Feature preview:")
display(df_features.head())

print("Feature summary preview:")
display(X_features.describe().T.head(25))

Feature-engineered dataset shape: (24000, 109)
X_features shape: (24000, 105)
y_labels shape: (24000,)
Number of metadata columns: 4
Number of feature columns: 105

Modulation distribution:


,modulation_type,num_signals
0,OOK,1000
1,4ASK,1000
2,8ASK,1000
3,BPSK,1000
4,QPSK,1000
5,8PSK,1000
6,16PSK,1000
7,32PSK,1000
8,16APSK,1000
9,32APSK,1000


SNR distribution:


,snr,num_signals
0,30,24000


Feature preview:


,signal_index,modulation_id,modulation_type,snr,mean_I,std_I,min_I,max_I,median_I,range_I,q25_I,q75_I,iqr_I,skew_I,kurtosis_I,mean_Q,std_Q,min_Q,max_Q,median_Q,range_Q,q25_Q,q75_Q,iqr_Q,skew_Q,kurtosis_Q,mean_abs_I,mean_abs_Q,std_ratio_I_Q,mean_abs_ratio_I_Q,corr_I_Q,mean_magnitude,std_magnitude,min_magnitude,max_magnitude,median_magnitude,range_magnitude,q25_magnitude,q75_magnitude,iqr_magnitude,skew_magnitude,kurtosis_magnitude,mean_power,std_power,min_power,max_power,median_power,total_energy,rms_power,peak_to_average_power_ratio,magnitude_cv,power_cv,mean_phase,std_phase,min_phase,max_phase,range_phase,circular_mean_phase,circular_resultant_length,mean_unwrapped_phase,std_unwrapped_phase,unwrapped_phase_range,mean_inst_freq,std_inst_freq,min_inst_freq,max_inst_freq,median_inst_freq,range_inst_freq,q25_inst_freq,q75_inst_freq,iqr_inst_freq,skew_inst_freq,kurtosis_inst_freq,mean_fft_magnitude,std_fft_magnitude,min_fft_magnitude,max_fft_magnitude,median_fft_magnitude,skew_fft_magnitude,kurtosis_fft_magnitude,mean_fft_power,std_fft_power,max_fft_power,total_fft_power,spectral_centroid,spectral_spread,spectral_entropy,spectral_flatness,dominant_freq,peak_fft_power_ratio,spectral_rolloff_85_freq,spectral_rolloff_95_freq,occupied_bandwidth_90,cov_I_I,cov_I_Q,cov_Q_I,cov_Q_Q,constellation_area_proxy,quadrant_1_ratio,quadrant_2_ratio,quadrant_3_ratio,quadrant_4_ratio,quadrant_balance_std,near_origin_ratio,far_origin_ratio,radius_mean,radius_std,radius_unique_rounded_count,phase_unique_rounded_count
0,0,0,OOK,30,0.739839,0.781117,-0.500421,2.351720,0.559564,2.852141,0.016489,1.457256,1.440768,0.323544,-1.242235,0.506905,0.530846,-0.393557,1.572671,0.393726,1.966227,0.010876,1.010403,0.999527,0.285936,-1.299135,0.816420,0.559229,1.471457,1.459902,0.991287,0.990590,0.845569,0.000709,2.776072,0.679575,2.775363,0.225010,1.780160,1.555150,0.475403,-1.216012,1.696255,2.084705,5.021242e-07,7.706573,0.461823,1736.965210,1.302404,4.543287,0.853601,1.229004,-0.109794,1.330908,-2.937152,3.103870,6.041022,0.607782,0.537871,18.856350,7.840054,37.833401,0.016814,0.704658,-3.140634,3.140934,0.000107,6.281568,-0.000887,0.001273,0.002161,0.423945,15.776016,11.120085,40.166016,0.015677,918.359863,0.385210,12.392648,254.662811,1736.965332,26543.375000,8.433848e+05,1778652.50,0.000024,0.024246,4.397296,0.000278,0.0,0.474171,0.023438,0.044922,0.089844,0.610740,0.411441,0.411441,0.282073,0.414653,0.768555,0.002930,0.228516,0.000000,0.313410,0.567383,0.230469,0.990590,0.845569,241,54
1,1,0,OOK,30,1.011886,1.002867,-0.734198,2.919468,0.918309,3.653667,0.078252,1.958826,1.880574,0.141994,-1.378221,-0.019308,0.033715,-0.137604,0.024156,-0.005056,0.161760,-0.031092,0.001897,0.032990,-1.501081,1.616800,1.109965,0.023951,29.745008,46.342579,-0.575323,1.110384,0.893427,0.001663,2.919661,0.919164,2.917998,0.239035,1.959959,1.720923,0.317480,-1.382836,2.031164,2.309860,2.766188e-06,8.524419,0.844862,2079.912109,1.425189,4.196815,0.804610,1.137210,0.451268,1.348481,-3.141350,3.141257,6.282607,-0.018225,0.582017,-4.837897,6.640921,25.169464,0.003007,0.652751,-3.140761,3.141482,-0.000053,6.282243,-0.000911,0.000736,0.001648,0.139158,19.739037,12.434528,43.878178,0.784873,1036.359619,1.538881,13.592319,290.584595,2079.912109,33805.968750,1.074041e+06,2129830.00,-0.000013,0.025126,4.101415,0.003429,0.0,0.504285,0.024414,0.043945,0.087891,1.006725,-0.019472,-0.019472,0.001138,0.033812,0.150391,0.179688,0.029297,0.640625,0.232467,0.535156,0.234375,1.110384,0.893427,242,28
2,2,0,OOK,30,0.811902,0.713866,-0.462071,2.154327,0.836580,2.616398,0.149772,1.465518,1.315745,-0.023391,-1.360287,0.798254,0.702855,-0.420834,2.130226,0.823946,2.551059,0.144123,1.462950,1.318827,-0.028206,-1.361135,0.874570,0.861073,1.015666,1.015675,0.971862,1.231032,0.885757,0.003267,2.771781,1.209571,2.768514,0.347216,2.074491,1.727275,0.119565,-1.521644,2.300004,2.336544,1.067332e-05,7.682770,1.463062,2355.204590,1.516577,3.340328,0.719524,1.015887,0.216069,1.205052,-2.493611,0.923082,3.416693,0.7

Feature summary preview:


,count,mean,std,min,25%,50%,75%,max
mean_I,24000.0,-0.009671,2.187662,-23.275093,-0.051530,0.000046,0.051457,26.465673
std_I,24000.0,0.665111,0.180014,0.005271,0.667016,0.704114,0.728993,1.258249
min_I,24000.0,-1.391574,2.219330,-25.749222,-1.661863,-1.498246,-1.190944,24.613077
max_I,24000.0,1.373605,2.220663,-20.400421,1.189623,1.498253,1.662746,28.459764
median_I,24000.0,-0.009875,2.189590,-23.275431,-0.075879,0.000023,0.074857,26.608673
range_I,24000.0,2.765179,0.800170,0.027416,2.365159,2.997433,3.266484,5.884624
q25_I,24000.0,-0.563113,2.196319,-23.855545,-0.693881,-0.570424,-0.462832,26.221281
q75_I,24000.0,0.543733,2.194167,-22.744829,0.461685,0.571042,0.694625,26.827707
iqr_I,24000.0,1.106847,0.354751,0.007232,0.992805,1.113613,1.272150,2.682431
skew_I,24000.0,0.002321,0.227008,-4.494058,-0.081421,0.000059,0.081380,2.727654
